In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
def sigmoid(x, a1, a2, a3, a4):
    s = 1 + np.exp( -a3*(x-a2) )
    y = a1 / s + a4
    return y

def inverse(x, a, b, c):
    return a/(x-b) + c

popt1 = [1.741759, 5.016247, 0.003036, -0.866097]
popt2 = [0.54774, -2.797004, 0.790583]
popt3 = [0.548174, 2.829586, -0.785972]

def eval_to_winodds(x):
    try:
        return sigmoid(float(x), *popt1)
    except:
        try:
            if float(x[1:]) > 0:
                return inverse(float(x[1:]), *popt2)
            elif float(x[1:]) < 0:
                return inverse(float(x[1:]), *popt3)
        except:
            return 0
    return 0

**Adding features**

In [ ]:
def generate_features(df_games, df_moves):
    
    # Prepare
    df_moves["IsInaccuracy"] = df_moves["Move"].str.endswith("?!")
    df_moves["IsBlunder"] = df_moves["Move"].str.endswith("??")
    df_moves["IsMistake"] = df_moves["Move"].str.endswith("?") & (~df_moves["Move"].str.endswith("??"))
    df_moves["IsWrongMove"] = df_moves["IsInaccuracy"] | df_moves["IsBlunder"] | df_moves["IsMistake"]
    df_moves["IsBadMove"] = df_moves["IsBlunder"] | df_moves["IsMistake"]
    df_moves["IsOkayMove"] = ~(df_moves["IsInaccuracy"] | df_moves["IsBlunder"] | df_moves["IsMistake"])

    df_moves["EvalCentipawn"] = pd.to_numeric(df_moves["Eval"], errors="coerce").multiply(100).round()
    df_moves["EvalCentipawn"] = df_moves["EvalCentipawn"].clip(-1200, 1200)

    df_moves["AbsEval"] = df_moves["EvalCentipawn"].abs()

    df_moves["CentipawnLoss"] = df_moves["EvalCentipawn"].diff()
    df_moves.loc[(df_moves["MoveNumber"] == 1), "CentipawnLoss"] = 0
    df_moves.loc[ df_moves["MoveSide"] == 0, "CentipawnLoss" ] = -1 * df_moves.loc[ df_moves["MoveSide"] == 0, "CentipawnLoss" ].clip(upper=0)
    df_moves.loc[ df_moves["MoveSide"] == 1, "CentipawnLoss" ] = df_moves.loc[ df_moves["MoveSide"] == 1, "CentipawnLoss" ].clip(lower=0)


    df_moves["Piece"] = df_moves["Move"].str[0]

    df_moves["IsCheck"] = df_moves["Move"].str.contains("+", regex=False)
    df_moves["IsCapture"] = df_moves["Move"].str.contains("x", regex=False)
    
    # Winodds
    df_moves["Winodds"] = df_moves["Eval"].apply(eval_to_winodds)

    df_moves["AdvLoss"] = df_moves["Winodds"].diff()
    df_moves.loc[(df_moves["MoveNumber"] == 1), "AdvLoss"] = 0
    df_moves.loc[ df_moves["MoveSide"] == 0, "AdvLoss" ] = -1 * df_moves.loc[ df_moves["MoveSide"] == 0, "AdvLoss" ].clip(upper=0)
    df_moves.loc[ df_moves["MoveSide"] == 1, "AdvLoss" ] = df_moves.loc[ df_moves["MoveSide"] == 1, "AdvLoss" ].clip(lower=0)
    

    # Main
    df_moves["StartLoss"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] <= 20)
    )

    df_moves["QueenLoss"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] <= 25) & (df_moves["Piece"] == "Q")
    )

    df_moves["RookLoss"] = df_moves["AdvLoss"].where( 
        (df_moves["Piece"] == "R")
    )
    
    df_moves["KnightLoss"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] <= 15) & (df_moves["Piece"] == "N")
    )

    df_moves["BishopLoss"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] <= 15) & (df_moves["Piece"] == "B")
    )

    df_moves["KingLoss1"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] >= 45) & (df_moves["Piece"] == "K")
    )

    df_moves["KingLoss1"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] >= 45) & (df_moves["Piece"] == "K")
    )

    df_moves["KingLoss2"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] <= 35) & (df_moves["Piece"] == "K")
    )

    df_moves["PawnLoss1"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] <= 20) & ( df_moves["Piece"].str.islower() )
    )

    df_moves["PawnLoss2"] = df_moves["AdvLoss"].where( 
        ( df_moves["Piece"].str.islower() )
    )

    df_moves["FlagPawnLoss1"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] <= 40) & ( df_moves["Piece"].isin(["a", "b", "g", "h"]) )
    )

    df_moves["FlagPawnLoss2"] = df_moves["AdvLoss"].where( 
        ( df_moves["Piece"].isin(["a", "b", "g", "h"]) )
    )

    df_moves["CenterPawnLoss1"] = df_moves["AdvLoss"].where( 
        ( df_moves["Piece"].isin(["c", "d", "e", "f"]) )
    )

    df_moves["CenterPawnLoss2"] = df_moves["AdvLoss"].where( 
        (df_moves["MoveNumber"] <= 30) & ( df_moves["Piece"].isin(["c", "d", "e", "f"]) )
    )

    features_tuples = [
        ("StartLoss", "mean"),
        ("QueenLoss", "mean"),
        ("RookLoss", "mean"),
        ("KnightLoss", "mean"),
        ("BishopLoss", "mean"),
        ("KingLoss1", "mean"),
        ("KingLoss2", "mean"),
        ("PawnLoss1", "mean"),
        ("PawnLoss2", "mean"),
        ("FlagPawnLoss1", "mean"),
        ("FlagPawnLoss2", "mean"),
        ("CenterPawnLoss1", "mean"),
        ("CenterPawnLoss2", "mean"),


    ]

    agg_dict = {
        f"{agg_func.capitalize()}{feature}": (feature, agg_func)
        for feature, agg_func in features_tuples
    }

    # !Агрегируем
    agg = df_moves.groupby("GameId").agg(**agg_dict)
    
    # NaN filling    
    agg = agg.fillna( agg.median() )
    
    get_mean_elo = lambda df: (df["WhiteElo"] + df["BlackElo"]) // 2

    df_features = (
        df_games
        .assign(Elo=get_mean_elo)
        .loc[:, ["GameId", "WhiteElo", "BlackElo", "Elo", "Opening", "ECO", "FirstMoves"]]
    )

    df_features = df_features.merge(agg, on="GameId")
    
    return df_features

In [ ]:
n_batches = 50
for batch in range(1, n_batches + 1):

    df_games = pd.read_parquet(f"01_parsed/batch_{batch}_games.parquet")
    df_moves = pd.read_parquet(f"01_parsed/batch_{batch}_moves.parquet")
        
    ids = set(df_games["GameId"]) & set(df_moves["GameId"])
    df_games = df_games[ df_games["GameId"].isin(ids) ]
    df_moves = df_moves[ df_moves["GameId"].isin(ids) ]
    
    # weird na values
    minor_last_move_error = df_moves[(
        df_moves["Move"].str.contains("#") &
        df_moves["Eval"].isna() &
        df_moves["Clock"].isna()
    )].index
    df_moves = df_moves.drop(index=minor_last_move_error)
    df_features = generate_features(df_games, df_moves)

    df_features.to_parquet(f"02_features/batch_{batch}.parquet")
    
    print(f"Batch {batch}: Done")